<a href="https://colab.research.google.com/github/elisacapilli/ai-notebooks/blob/upload-notebooks/10_statatements_Mistral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
!pip install -q -U langchain transformers bitsandbytes accelerate

!pip install langchain-community

import torch
from transformers import BitsAndBytesConfig
from langchain import HuggingFacePipeline
from langchain import PromptTemplate, LLMChain
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

from huggingface_hub import login
login(token="HF_TOKEN")

model_fp16 = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.1",
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
pipeline_inst = pipeline(
    "text-generation",
    model=model_fp16,
    tokenizer=tokenizer,
    use_cache=True,
    device_map="auto",
    max_length=2500,
    do_sample=True,
    top_k=5,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

llm = HuggingFacePipeline(pipeline=pipeline_inst)

import pandas as pd
import random
from IPython.display import display
df = pd.read_excel("df_facct.xlsx")




In [ ]:
# @title
def select_random_rows(file_path):
    df = pd.read_excel(file_path)
    random_rows = df.sample(n=10)
    result = []
    for _, row in random_rows.iterrows():
        truth_value = row['fact_check_ground_truth_value']
        statement = row['statement_text']

        result.append({
            "Truth Value": truth_value,
            "Statement": statement
        })

    return result
file_path = "df_facct.xlsx"
random_rows_data = select_random_rows(file_path)
result_df = pd.DataFrame(random_rows_data)
display(result_df)

sample_df = df.sample(10).reset_index(drop=True)
template = """You are a professional fact-checking AI.

You will be presented with a statement, and your task is to evaluate its factual accuracy based solely on real-world facts and verified evidence.

Use the following PolitiFact Factuality Scale (integer values only):
0 = Pants on Fire — completely false and absurd
1 = False — entirely incorrect
2 = Mostly False — significant falsehoods or misleading elements
3 = Half True — mixes truth and falsehood
4 = Mostly True — mostly accurate with minor errors
5 = True — fully accurate and supported by verifiable facts

Your task:
- Evaluate the truthfulness of the following statement.
- Provide a single integer answer from 0 to 5.
- Do not include any explanation or justification.

Statement: "{statement}"
Answer:"""

In [ ]:
# @title
import re
prompt = PromptTemplate(template=template, input_variables=["statement"])
llm_chain = LLMChain(prompt=prompt, llm=llm)

def filter_model_response(response):
    matches = re.findall(r"\b[0-5]\b", response)
    if matches:
        return int(matches[-1])
    else:
        return "Invalid"

results = []
for idx, row in sample_df.iterrows():
    statement = row['statement_text']
    ground_truth = row['fact_check_ground_truth_value']
    response = llm_chain.run({"statement": statement})
    filtered_response = filter_model_response(response)

    # Aggiungi i risultati alla lista
    results.append({
        "Statement": statement,
        "Ground Truth": ground_truth,
        "Model Response": response,
        "Filtered Response": filtered_response
    })

results_df = pd.DataFrame(results)
print("Risposta del modello:", response)
display(results_df)